# 🍄 Mushroom Image Dataset Expander

This notebook downloads real mushroom images from **Bing Image Search** to expand the dataset to a target of **50 images per category**.

### Features:
- Automatically detects how many images each category needs
- Downloads real, diverse images from the web
- Validates images (removes corrupt/broken files)
- Skips categories that already have enough images
- Deduplicates images using MD5 hashing
- Resume-friendly (saves progress to JSON)
- Logs all activity

## 1️⃣ Install Dependencies

In [ ]:
!pip install icrawler pillow requests

## 2️⃣ Import Libraries

In [ ]:
import os
import sys
import time
import json
import shutil
import hashlib
import logging
from pathlib import Path
from datetime import datetime

from PIL import Image
from icrawler.builtin import BingImageCrawler, GoogleImageCrawler

print("✅ All libraries imported successfully!")

## 3️⃣ Configuration

⚙️ **Edit these settings as needed:**

In [ ]:
# ============================================================
# CONFIGURATION - Edit these as needed
# ============================================================

DATASET_DIR = "/home/ashirkhan/Updated Data set/raw_images_renamed"
TARGET_IMAGES = 50              # Target number of images per category
EXTRA_DOWNLOAD = 15             # Download extra to account for invalid/duplicate removals
MIN_IMAGE_SIZE = (50, 50)       # Minimum image dimensions (width, height)
MAX_IMAGE_SIZE_BYTES = 15 * 1024 * 1024  # 15 MB max file size
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}
DELAY_BETWEEN_CATEGORIES = 3    # Seconds between categories to avoid rate limiting

# Search query templates for better & diverse results
SEARCH_TEMPLATES = [
    "{name} mushroom",
    "{name} fungus",
    "{name} mushroom identification",
]

print(f"📁 Dataset directory: {DATASET_DIR}")
print(f"🎯 Target images per category: {TARGET_IMAGES}")
print(f"📥 Extra downloads per category: {EXTRA_DOWNLOAD}")

## 4️⃣ Setup Logging

In [ ]:
# Setup logging to both file and notebook output
log_file = os.path.join(os.path.dirname(DATASET_DIR), "download_log.txt")

# Remove existing handlers to avoid duplicates when re-running
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

print(f"📝 Log file: {log_file}")

## 5️⃣ Helper Functions

In [ ]:
def get_image_count(folder_path):
    """Count valid image files in a folder."""
    count = 0
    for f in os.listdir(folder_path):
        ext = os.path.splitext(f)[1].lower()
        if ext in VALID_EXTENSIONS:
            count += 1
    return count


def get_existing_hashes(folder_path):
    """Get MD5 hashes of existing images to avoid duplicates."""
    hashes = set()
    for f in os.listdir(folder_path):
        filepath = os.path.join(folder_path, f)
        ext = os.path.splitext(f)[1].lower()
        if ext in VALID_EXTENSIONS and os.path.isfile(filepath):
            try:
                with open(filepath, 'rb') as fh:
                    file_hash = hashlib.md5(fh.read()).hexdigest()
                    hashes.add(file_hash)
            except Exception:
                pass
    return hashes


def validate_image(filepath):
    """
    Validate that a file is a valid, non-corrupt image.
    Returns True if valid, False otherwise.
    """
    try:
        with Image.open(filepath) as img:
            img.verify()
        
        # Re-open to check dimensions (verify() doesn't allow further ops)
        with Image.open(filepath) as img:
            width, height = img.size
            if width < MIN_IMAGE_SIZE[0] or height < MIN_IMAGE_SIZE[1]:
                return False
        
        # Check file size
        file_size = os.path.getsize(filepath)
        if file_size > MAX_IMAGE_SIZE_BYTES or file_size < 1000:  # < 1KB likely corrupt
            return False
        
        return True
    except Exception:
        return False


def clean_name_for_search(folder_name):
    """Convert folder name to a nice search query."""
    return folder_name.replace('_', ' ').replace('-', ' ').strip()


def get_next_image_number(folder_path):
    """Get the next available image number for naming."""
    max_num = 0
    for f in os.listdir(folder_path):
        name_part = os.path.splitext(f)[0]
        parts = name_part.split('_')
        for part in reversed(parts):
            try:
                num = int(part)
                max_num = max(max_num, num)
                break
            except ValueError:
                continue
    return max_num + 1


def get_all_categories(dataset_dir):
    """Get all mushroom category folders and their current image counts."""
    categories = {}
    for entry in sorted(os.listdir(dataset_dir)):
        full_path = os.path.join(dataset_dir, entry)
        if os.path.isdir(full_path):
            count = get_image_count(full_path)
            categories[entry] = {
                'path': full_path,
                'current_count': count,
                'needed': max(0, TARGET_IMAGES - count)
            }
    return categories


def save_progress(progress_file, completed):
    """Save progress to a JSON file for resume capability."""
    with open(progress_file, 'w') as f:
        json.dump(completed, f, indent=2)


def load_progress(progress_file):
    """Load progress from a JSON file."""
    if os.path.exists(progress_file):
        with open(progress_file, 'r') as f:
            return json.load(f)
    return {}


print("✅ Helper functions defined!")

## 6️⃣ Download Function (Core Logic)

In [ ]:
def download_images_for_category(category_folder, category_name, num_needed):
    """
    Download images for a single mushroom category using Bing Image Search.
    
    Args:
        category_folder: Full path to the category's image folder
        category_name: Clean name of the mushroom (e.g., 'chanterelle')
        num_needed: Number of additional images to download
    
    Returns:
        Number of valid images successfully added
    """
    # Create a temporary download folder
    temp_dir = os.path.join(os.path.dirname(DATASET_DIR), "temp_downloads", category_name)
    os.makedirs(temp_dir, exist_ok=True)
    
    search_name = clean_name_for_search(category_name)
    total_to_download = num_needed + EXTRA_DOWNLOAD  # Download extra for safety
    
    # Get existing hashes to avoid duplicates
    existing_hashes = get_existing_hashes(category_folder)
    
    # Try downloading with Bing
    downloaded_count = 0
    
    for template in SEARCH_TEMPLATES:
        if downloaded_count >= total_to_download:
            break
            
        query = template.format(name=search_name)
        remaining = total_to_download - downloaded_count
        
        logger.info(f"  Searching: '{query}' (need {remaining} more)")
        
        try:
            # Suppress icrawler's verbose logging
            crawler_logger = logging.getLogger('icrawler')
            crawler_logger.setLevel(logging.WARNING)
            
            crawler = BingImageCrawler(
                storage={'root_dir': temp_dir},
                downloader_threads=4,
                log_level=logging.WARNING
            )
            
            crawler.crawl(
                keyword=query,
                max_num=remaining,
                min_size=(100, 100),
                file_idx_offset=downloaded_count
            )
            
            # Count what we got
            downloaded_count = len([f for f in os.listdir(temp_dir) if os.path.isfile(os.path.join(temp_dir, f))])
            
        except Exception as e:
            logger.warning(f"  Bing search failed for '{query}': {e}")
            
            # Fallback to Google
            try:
                logger.info(f"  Trying Google fallback for: '{query}'")
                crawler = GoogleImageCrawler(
                    storage={'root_dir': temp_dir},
                    downloader_threads=4,
                    log_level=logging.WARNING
                )
                crawler.crawl(
                    keyword=query,
                    max_num=remaining,
                    min_size=(100, 100),
                    file_idx_offset=downloaded_count
                )
                downloaded_count = len([f for f in os.listdir(temp_dir) if os.path.isfile(os.path.join(temp_dir, f))])
            except Exception as e2:
                logger.warning(f"  Google fallback also failed: {e2}")
    
    # Now validate, deduplicate, and move good images
    valid_added = 0
    next_num = get_next_image_number(category_folder)
    
    for filename in sorted(os.listdir(temp_dir)):
        filepath = os.path.join(temp_dir, filename)
        
        if not os.path.isfile(filepath):
            continue
        
        # Validate the image
        if not validate_image(filepath):
            logger.debug(f"  Invalid image removed: {filename}")
            os.remove(filepath)
            continue
        
        # Check for duplicates via hash
        with open(filepath, 'rb') as fh:
            file_hash = hashlib.md5(fh.read()).hexdigest()
        
        if file_hash in existing_hashes:
            logger.debug(f"  Duplicate skipped: {filename}")
            os.remove(filepath)
            continue
        
        # Convert to JPG and move to category folder
        try:
            with Image.open(filepath) as img:
                # Convert to RGB if necessary (handles RGBA, P mode, etc.)
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                
                new_filename = f"{category_name}_{next_num:04d}.jpg"
                new_filepath = os.path.join(category_folder, new_filename)
                
                img.save(new_filepath, 'JPEG', quality=90)
                
                existing_hashes.add(file_hash)
                valid_added += 1
                next_num += 1
                
        except Exception as e:
            logger.debug(f"  Error processing {filename}: {e}")
        
        os.remove(filepath)
        
        # Stop if we have enough
        if valid_added >= num_needed:
            break
    
    # Clean up temp directory
    try:
        shutil.rmtree(temp_dir)
    except Exception:
        pass
    
    return valid_added


print("✅ Download function defined!")

## 7️⃣ Analyze Current Dataset

Let's see how many images each mushroom category currently has and how many we need to download.

In [ ]:
# Analyze dataset
categories = get_all_categories(DATASET_DIR)

needs_download = {k: v for k, v in categories.items() if v['needed'] > 0}
already_done = {k: v for k, v in categories.items() if v['needed'] == 0}
total_needed = sum(v['needed'] for v in needs_download.values())

print("=" * 60)
print("📊 DATASET ANALYSIS")
print("=" * 60)
print(f"  Total categories:                {len(categories)}")
print(f"  Already at target (≥{TARGET_IMAGES}):     {len(already_done)}")
print(f"  Need more images:                {len(needs_download)}")
print(f"  Total images to download:        ~{total_needed}")
print("=" * 60)

print(f"\n📋 Categories that need more images (sorted by count):")
print("-" * 60)
for name, info in sorted(needs_download.items(), key=lambda x: x[1]['current_count']):
    bar = '█' * info['current_count'] + '░' * info['needed']
    print(f"  {name:<35} {info['current_count']:>3} → {TARGET_IMAGES} (need {info['needed']})")

## 8️⃣ Run Download - ALL Categories

⚠️ **This will download images for ALL categories that need more images.**  
📌 This may take a while depending on your internet speed (~215 categories).  
🔄 **Resume-friendly**: If the notebook stops, just re-run this cell to continue where it left off.

In [ ]:
# ============================================================
# 🚀 DOWNLOAD ALL CATEGORIES
# ============================================================

# Reload categories (in case you ran the analysis cell earlier)
categories = get_all_categories(DATASET_DIR)
needs_download = {k: v for k, v in categories.items() if v['needed'] > 0}

# Progress tracking
progress_file = os.path.join(os.path.dirname(DATASET_DIR), "download_progress.json")
progress = load_progress(progress_file)

# Temp directory
temp_base = os.path.join(os.path.dirname(DATASET_DIR), "temp_downloads")
os.makedirs(temp_base, exist_ok=True)

# Counters
successful = 0
failed = 0
skipped = 0

print("=" * 70)
print("🍄 STARTING MUSHROOM IMAGE DOWNLOAD")
print(f"   Categories to process: {len(needs_download)}")
print(f"   Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

for idx, (name, info) in enumerate(sorted(needs_download.items()), 1):
    # Check if already completed in a previous run
    if name in progress and progress[name].get('completed', False):
        current = get_image_count(info['path'])
        if current >= TARGET_IMAGES:
            print(f"[{idx}/{len(needs_download)}] ⏭️  {name} - Already completed (has {current})")
            skipped += 1
            continue
    
    print(f"\n[{idx}/{len(needs_download)}] 🍄 Processing: {name}")
    print(f"   Current: {info['current_count']} images | Need: {info['needed']} more")
    
    try:
        added = download_images_for_category(
            category_folder=info['path'],
            category_name=name,
            num_needed=info['needed']
        )
        
        new_total = get_image_count(info['path'])
        
        if added > 0:
            print(f"   ✅ Added {added} images → Total now: {new_total}")
            successful += 1
        else:
            print(f"   ⚠️  No valid images downloaded")
            failed += 1
        
        # Save progress
        progress[name] = {
            'completed': new_total >= TARGET_IMAGES,
            'images_added': added,
            'total_images': new_total,
            'timestamp': datetime.now().isoformat()
        }
        save_progress(progress_file, progress)
        
    except Exception as e:
        print(f"   ❌ Error processing {name}: {e}")
        failed += 1
        progress[name] = {
            'completed': False,
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        }
        save_progress(progress_file, progress)
    
    # Delay between categories to avoid rate limiting
    if idx < len(needs_download):
        time.sleep(DELAY_BETWEEN_CATEGORIES)

# Clean up temp directory
try:
    shutil.rmtree(temp_base)
except Exception:
    pass

print("\n" + "=" * 70)
print("📋 DOWNLOAD COMPLETE!")
print("=" * 70)
print(f"   ✅ Successful: {successful} categories")
print(f"   ⏭️  Skipped (already done): {skipped} categories")
print(f"   ❌ Failed: {failed} categories")
print(f"   Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

## 🔧 (Alternative) Run Download - SPECIFIC Categories

Use this cell if you want to download images for **specific mushroom categories only**.  
Edit the `selected_categories` list below.

In [ ]:
# ============================================================
# 🎯 DOWNLOAD SPECIFIC CATEGORIES ONLY
# ============================================================
# Edit this list with the categories you want to download:

selected_categories = [
    "chanterelle",
    "fly_agaric",
    "truffles",
    # Add more category folder names here...
]

# Progress tracking
progress_file = os.path.join(os.path.dirname(DATASET_DIR), "download_progress.json")
progress = load_progress(progress_file)

for idx, category_name in enumerate(selected_categories, 1):
    category_path = os.path.join(DATASET_DIR, category_name)
    
    if not os.path.isdir(category_path):
        print(f"[{idx}/{len(selected_categories)}] ❌ Folder not found: {category_name}")
        continue
    
    current_count = get_image_count(category_path)
    needed = max(0, TARGET_IMAGES - current_count)
    
    if needed == 0:
        print(f"[{idx}/{len(selected_categories)}] ⏭️  {category_name} already has {current_count} images")
        continue
    
    print(f"\n[{idx}/{len(selected_categories)}] 🍄 Processing: {category_name}")
    print(f"   Current: {current_count} | Need: {needed} more")
    
    try:
        added = download_images_for_category(
            category_folder=category_path,
            category_name=category_name,
            num_needed=needed
        )
        new_total = get_image_count(category_path)
        print(f"   ✅ Added {added} images → Total now: {new_total}")
        
        progress[category_name] = {
            'completed': new_total >= TARGET_IMAGES,
            'images_added': added,
            'total_images': new_total,
            'timestamp': datetime.now().isoformat()
        }
        save_progress(progress_file, progress)
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
    
    if idx < len(selected_categories):
        time.sleep(DELAY_BETWEEN_CATEGORIES)

print("\n✅ Done with selected categories!")

## 9️⃣ Final Dataset Report

Run this cell after downloading to see the final status of your dataset.

In [ ]:
# ============================================================
# 📊 FINAL DATASET REPORT
# ============================================================

categories_final = get_all_categories(DATASET_DIR)

below_target = []
at_target = []
above_target = []
total_images = 0

for name, info in sorted(categories_final.items()):
    total_images += info['current_count']
    if info['current_count'] < TARGET_IMAGES:
        below_target.append((name, info['current_count']))
    elif info['current_count'] == TARGET_IMAGES:
        at_target.append((name, info['current_count']))
    else:
        above_target.append((name, info['current_count']))

print("=" * 70)
print("📊 FINAL DATASET REPORT")
print("=" * 70)
print(f"  Total categories:         {len(categories_final)}")
print(f"  Total images:             {total_images}")
print(f"  Avg images per category:  {total_images / len(categories_final):.1f}")
print(f"  ✅ At target ({TARGET_IMAGES}):       {len(at_target)}")
print(f"  🟢 Above target:          {len(above_target)}")
print(f"  🔴 Below target:          {len(below_target)}")
print("=" * 70)

if below_target:
    print(f"\n⚠️  Categories still below target ({len(below_target)}):")
    print("-" * 50)
    for name, count in sorted(below_target, key=lambda x: x[1]):
        deficit = TARGET_IMAGES - count
        bar = '█' * count + '░' * deficit
        print(f"  {name:<35} {count:>3}/{TARGET_IMAGES} (need {deficit})")
    print(f"\n💡 TIP: Re-run the download cell to retry failed categories")
    print(f"💡 TIP: Or use image augmentation to fill remaining gaps")
else:
    print(f"\n🎉 ALL categories have reached {TARGET_IMAGES}+ images!")
    print("Your dataset is ready for training!")

## 🔟 View Download Progress (Optional)

Check the saved progress file to see details of each category's download status.

In [ ]:
# View download progress
progress_file = os.path.join(os.path.dirname(DATASET_DIR), "download_progress.json")

if os.path.exists(progress_file):
    with open(progress_file, 'r') as f:
        progress = json.load(f)
    
    completed = sum(1 for v in progress.values() if v.get('completed', False))
    failed = sum(1 for v in progress.values() if 'error' in v)
    
    print(f"📊 Progress Summary:")
    print(f"   Processed: {len(progress)} categories")
    print(f"   Completed: {completed}")
    print(f"   Failed: {failed}")
    
    if failed > 0:
        print(f"\n❌ Failed categories:")
        for name, info in progress.items():
            if 'error' in info:
                print(f"   {name}: {info['error']}")
else:
    print("No progress file found. Run the download first!")